# Mission 08: Awesome Agent Skills & Dynamic Skill Loading

이 노트북은 Vercel Labs의 **`npx skills`** CLI와 **awesome-agent-skills** 오픈소스 레지스트리를 연계하여, 깃허브 상에 공개된 검증된 스킬셋을 한 줄의 명령어로 추가하고 우리 에이전트가 이를 자율적으로 학습해 기동하도록 연동하는 실습입니다.

In [ ]:
# 1. 환경 변수 및 패키지 탐색 경로 로드
import sys
import os
from dotenv import load_dotenv

while not os.path.exists("app") and os.getcwd() != "/":
    os.chdir("..")
sys.path.append(os.path.abspath("src"))
sys.path.append(os.path.abspath("."))
load_dotenv(override=True)

print(f"📌 현재 작업 디렉토리: {os.getcwd()}")

### [단계 1] 스킬 매니저 CLI 작동 확인 및 스킬 다운로드

프로젝트 터미널을 열고 다음 명령어를 실행하여 OpenAI의 **`jupyter-notebook`** 생성 스킬을 로컬에 다운로드합니다:

```bash
npx -y skills add openai/jupyter-notebook
```

명령이 정상적으로 수행되면 프로젝트 루트 디렉토리 하위에 **`skills/jupyter-notebook/`** 폴더가 생성되고, 그 아래에 **`Skill.md`** 가이드와 관련 헬퍼 스크립트들이 복사됩니다.

In [ ]:
# 다운로드된 스킬 파일 구조 확인
skill_dir = "skills/jupyter-notebook"
if os.path.exists(skill_dir):
    print(f"✅ 스킬 '{skill_dir}' 다운로드 확인 완료!")
    print(f"📁 디렉토리 구조: {os.listdir(skill_dir)}")
else:
    print(f"⚠️ {skill_dir} 경로가 존재하지 않습니다. 터미널에서 'npx -y skills add openai/jupyter-notebook' 명령어를 먼저 구동해 주세요.")

### [단계 2] 기본 도구(Bash/FileRead)만을 장착한 에이전트의 공개 스킬 학습 구동

에이전트에게 처음부터 `jupyter-notebook` 생성 도구를 직접 주입하는 것이 아닙니다. 에이전트에게는 오직 파일 읽기(`file_read`)와 리눅스 셸 명령 실행(`bash_command`)이라는 2가지 기본 도구만 쥐어준 상태에서, 프롬프트에 `Progressive Skill Disclosure` 지침을 주어 로컬 스킬을 학습하여 실행하도록 제어합니다.

In [ ]:
from harness.tools.claude_tools import file_read, bash_command
from app.utils import get_llm
from langchain.agents import create_agent

# 1. 초경량 Gemini Flash 모델 로드 및 오직 2가지 기본 도구만 바인딩
llm = get_llm(model_name="gemini-3.5-flash", temperature=0.0)
base_tools = [file_read, bash_command]

# 2. 에이전트가 로컬 skills 디렉토리를 자율적으로 우선 탐색하도록 가이드
system_instruction = (
    "You are a progressive skill-disclosure agent.\n"
    "You only have 'file_read' and 'bash_command' base tools.\n"
    "Your objective is to solve the user request by discovering and executing custom scripts under the local 'skills/' directory.\n\n"
    "Guidelines:\n"
    "1. ALWAYS prioritize searching and utilizing custom scripts under the 'skills/' folder over running raw shell commands directly in 'bash_command'.\n"
    "2. First, read 'skills/jupyter-notebook/Skill.md' using 'file_read' to learn how to run the corresponding python script and its arguments.\n"
    "3. Execute the script via 'bash_command' using the exact python path and parameters to create the requested file.\n"
    "Finally, report the result clearly to the user."
)

# 3. ReAct 에이전트 빌드
from langgraph.checkpoint.memory import MemorySaver
checkpointer = MemorySaver()
skill_agent = create_agent(
    model=llm,
    tools=base_tools,
    system_prompt=system_instruction,
    checkpointer=checkpointer
)

print("🚀 Progressive Skill 에이전트 기동 완료! 자율 탐색 및 실행을 준비합니다.")

### [단계 3] 공개 스킬 활용 격발 및 결과물 검증

사용자는 단지 노트북 생성을 지시하고, 에이전트가 `Skill.md`를 찾아 읽은 뒤 파이썬 명령어로 노트북을 생성해 내는지 디버거를 기동하여 핑퐁 궤적을 실시간으로 관찰합니다.

In [ ]:
from utils.test_log import stream_and_debug_agent
from langchain_core.messages import HumanMessage

# 에이전트 구동 지시
user_request = (
    "skills/jupyter-notebook/Skill.md 파일에 적힌 사용법과 가이드를 파악해줘. "
    "그 다음 지침에 따라 1부터 100까지의 합을 구하는 파이썬 코드가 적힌 주피터 노트북 파일 "
    "'artifacts/sum_1_to_100.ipynb'를 생성해줘."
)

config = {"configurable": {"thread_id": "skills_public_harness_session"}}
inputs = {"messages": [HumanMessage(content=user_request)]}

try:
    # 실시간 에이전트 자율 탐색 궤적 추적
    stream_and_debug_agent(skill_agent, inputs, config, agent_name="Progressive Skill 에이전트")
    
    # 최종 결과물 생성 여부 검증
    target_file = "artifacts/sum_1_to_100.ipynb"
    if os.path.exists(target_file):
        print(f"\n🎉 [검증 성공] 공개 스킬을 사용해 '{target_file}' 파일이 자율적으로 생성되었습니다!")
    else:
        print(f"\n❌ [검증 실패] '{target_file}' 파일이 생성되지 않았습니다. 에이전트 궤적을 점검하세요.")
except Exception as e:
    print(f"❌ 에러 발생: {e}")